In [3]:
import whisper
import os
import contextlib
with open(os.devnull, "w") as fnull:
    with contextlib.redirect_stderr(fnull):
        import pyaudio
        import speech_recognition as sr
import wave
import sys

from ollama import Client
from ollama import chat 

client = Client(headers={'Authorization': f"Bearer {os.getenv('OLLAMA_API_KEY')}"})

key = os.getenv("OLLAMA_API_KEY")

print("Key exists:", key is not None)
print("Key length:", len(key) if key else 0)

Key exists: True
Key length: 57


In [4]:
model = whisper.load_model("base")
result = model.transcribe("test.wav")
print(result['text'])

 1, 2, 3, 4, 1, 2, 3, 4, just then...


## Hyperparameter

In [5]:
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 2
RATE = 44100
RECORD_SECONDS = 7
WAVE_OUTPUT_FILENAME = "for_whisper.wav"

##  Code for recording using Pyaudio

Record is saved as "for_whisper.wav"

In [6]:
p = pyaudio.PyAudio()

stream = p.open(channels=CHANNELS, 
                rate=RATE, 
                format=FORMAT, 
                frames_per_buffer=CHUNK, 
                input=True)

print("*** RECORDING ***")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK)
    frames.append(data)

print("*** Done recording ***")

stream.stop_stream()
stream.close()
p.terminate()

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
wf.setnchannels(CHANNELS)
wf.setsampwidth(p.get_sample_size(FORMAT))
wf.setframerate(RATE)
wf.writeframes(b''.join(frames))
wf.close()

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5705:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM sysdefault
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_concat returned error: No

*** RECORDING ***
*** Done recording ***


## Testing Whisper .transcribe()
Model transcribes speech to text.

The audio data is the recording obtained above.

In [4]:
trans = model.transcribe("for_whisper.wav")
print(trans['text'])

 Because this house is not a home without my baby


## Simple Web search and fetch with Ollama

In [3]:

response = client.web_search("Latest AI tech news?")
link = response['results'][0]['url']

summ = client.web_fetch(link)

print(summ['content'])

Bill Gates says we’ve passed AI’s danger thresholds. Now what? | MIT Technology Review

Skip to Content

It’s a glorious day in Kirkland, Washington, an affluent Seattle suburb on the eastern shore of Lake Washington. The temperature is in the mid-80s, and the sky is incapable of being any more blue. The view from the Gates Ventures conference room overlooks the Carillon Point Marina, where a flotilla of expensive boats bob in the water, and across the lake to the Olympic Mountains that define the horizon. It’s gorgeous. And vaguely terrifying.

Because if the scene is placid, the messenger is not. Seated across from me at a conference room table, Bill Gates is rocking back and forth in his chair, totally animated. And the more he has to say—about the threats of terror or economic collapse or just losing control of our AI systems—the more agitated I find myself becoming, too.

The philanthropist and former Microsoft CEO says he has been growing increasingly alarmed by the rate of chang

## Testing the model's ability to summarize a piece of news

In [8]:
content = "Summarize this into a short text, keep the key information and do not make up any non-mentionned information. \n" + summ['content']
final = chat(model = 'qwen3:4b', messages=[{'role': 'user', 'content': f'{content}'}], stream=True)

for i in final:
    print(i['message']['content'], end='')

Here's a concise, focused summary of the key points from the Bill Gates/MIT Technology Review interview excerpt, structured for clarity and relevance:

---

### **Core Themes & Key Takeaways**
1. **The Human vs. AI Transition Challenge**  
   - Gates emphasizes that rapid AI advancement (e.g., coding, cyber capabilities) is outpacing societal readiness for ethical governance. **Example**: AI systems like Claude recently crossed a "massive cyberattack threshold" (e.g., potential to sabotage aircraft navigation), but real-world consequences were minimal—highlighting the gap between technical capability and responsible deployment.

2. **"Human Reserve" as a Policy Solution**  
   - To address job displacement (e.g., truck drivers, machine tool operators), Gates proposes **temporary "human reserves"**—a policy where workers are protected for 10+ years during transitions. This avoids abrupt shifts (e.g., forcing a 53-year-old to switch to childcare). *Purpose*: Either permanent preservation